## Task 3

In [9]:
import numpy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from nltk.translate.bleu_score import corpus_bleu
from collections import Counter

#### Part 1

In [2]:
def scaled_dot_product_attention(Q, K, V):
    ''' 
    Q is a matrix containing the query vectors. These represent the set of items you want to draw attention to
    K is a matrix containing the key vectors. Keys are paired with the values and are used to retrieve information.
        The model can use the similarity between a query and a key to determine how much attention to pay to the corresponding value.
    V is a matrix containing the value vectors. Values hold the actual information the model wants to retrieve. 
    
    Once the model determines which key and values are most relevant to a given query,
    it aggregates these values, weighted by their relevance, to produce the output.
    '''

    # Get the dimension of the keys (d_k). 
    # Q and K have shape (batch_size, seq_length, d_k).
    d_k = K.shape[-1]
    
    # Calculate the dot product of Q and the transpose of K.
    # (batch_size, seq_length, d_k) * (batch_size, d_k, seq_length)
    # Resulting shape: (batch_size, seq_length, seq_length) -> these are the raw attention scores.
    scores = Q @ K.transpose(0, 2, 1)    
    scaled_scores = scores / np.sqrt(d_k)
        
    # Apply the softmax function to get attention weights (probabilities)
    max_scores = np.max(scaled_scores, axis=-1, keepdims=True)
    exp_scores = np.exp(scaled_scores - max_scores)
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # Multiply the attention weights by the Values (V).
    # Resulting shape: (batch_size, seq_length, d_v)
    output = attention_weights @ V
    
    return output

#### Part 2

In [3]:
# Pytorch implementation of Part 1
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, query, keys, values):
        # query: (batch_size, 1, hidden_dim) -> from decoder
        # keys/values: (batch_size, seq_len, hidden_dim) -> from encoder
        
        d_k = query.size(-1)
        
        # Calculate scores: Q * K^T
        # keys.transpose(1, 2) changes shape to (batch_size, hidden_dim, seq_len)
        scaled_scores = torch.bmm(query, keys.transpose(1, 2)) / np.sqrt(d_k)
        
        # Attention weights
        attention_weights = F.softmax(scaled_scores, dim=-1) # (batch_size, 1, seq_len)
        
        # Context vector: weights * V
        output = torch.bmm(attention_weights, values) # (batch_size, 1, hidden_dim)
        
        return output


# ENCODER: Processes the input sentence word by word and outputs a sequence of vectors 
# (one for each word). It also outputs a final hidden state for the decoder.

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        
        # input_dim: Size of the source vocabulary (e.g., 10,000 unique words)
        # emb_dim: How many numbers we use to represent a word
        # hidden_dim: The size of the GRU's internal memory
        
        # Converts integer word IDs into dense meaning vectors.
        self.embedding = nn.Embedding(input_dim, emb_dim)
        
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, src):
        # Let's imagine an example batch:
        # We have 32 sentences in our batch (batch_size = 32).
        # The longest sentence is 10 words long (seq_len = 10).
        # src shape: (32, 10) -> A grid of 32 rows and 10 columns filled with integer word IDs.

        # Step 1: Embed the words
        # We pass the integers through the embedding layer and apply dropout.
        # Now, every single word ID (integer) is replaced by a vector of size 'emb_dim'
        embedded = self.dropout(self.embedding(src))
        # embedded shape: (32, 10, 256) -> (batch_size, seq_len, emb_dim)

        # Step 2: Pass through the GRU
        # The GRU reads the embedded words sequentially from left to right.
        outputs, hidden = self.gru(embedded)
        
        # 'outputs' contains the top-layer hidden state from the GRU for EVERY word in the sentence.
        # Because we are using Attention later, we need ALL of these to act as our Keys and Values.
        # outputs shape: (32, 10, 512) -> (batch_size, seq_len, hidden_dim)
        
        # 'hidden' is just the VERY LAST hidden state from the final word in the sentence.
        # We use this to initialize the Decoder's first step so it knows what sentence it's translating.
        # hidden shape: (1, 32, 512) -> (number_of_layers, batch_size, hidden_dim)

        return outputs, hidden

# DECODER: Generates the target sentence word by word. Instead of relying on a single 
# compressed vector, it uses an Attention Mechanism to look back at ALL the encoder's 
# word vectors and focuses on the most relevant ones to predict the next word.

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        
        # output_dim: Size of the TARGET vocabulary (e.g., 15,000 French words)
        
        self.embedding = nn.Embedding(output_dim, emb_dim)
        
        # Decoder GRU
        # Notice the input size: emb_dim + hidden_dim because we are going to concatenate the embedded word AND the context vector
        self.gru = nn.GRU(emb_dim + hidden_dim, hidden_dim, batch_first=True)
        
        # The Classifier (Linear Layer), take the GRU's output and the context vector and expands 
        # it back up to the size of our entire target dictionary to predict the next word
        self.fc_out = nn.Linear(hidden_dim * 2, output_dim)
        
    
        self.dropout = nn.Dropout(p=0.3)
        self.attention = ScaledDotProductAttention()

    def forward(self, input, hidden, encoder_outputs):
        # Let's say we are translating the 3rd word in the sentence. 
        # batch_size = 32 and hidden_dim = 512 and longest_seq is 10 words long.
        # input: The integer ID of the previous word generated. Shape: (32, 1)
        # hidden: Shape: (1, 32, 512)
        # encoder_outputs: Shape: (32, 10, 512)
        
        # Embed the current input word
        embedded = self.dropout(self.embedding(input)) # assuming emb_dim = 256
        # embedded shape: (32, 1, 256)
        
        # Prepare the Query for Attention
        # The attention block expects the batch size to be the first dimension.
        # So we transpose the hidden state from (1, 32, 512) to (32, 1, 512)
        query = hidden.transpose(0, 1) 
        
        # Calculate Attention
        context = self.attention(query, encoder_outputs, encoder_outputs)
        # context shape: (32, 1, 512)
        
        # Combine the input word and the context vector
        # (32, 1, 256) concatenated with (32, 1, 512) = (32, 1, 768)
        gru_input = torch.cat((embedded, context), dim=2)
        
        # Update the Decoder's hidden state
        output, hidden = self.gru(gru_input, hidden)
        # output shape: (32, 1, 512) -> The new top-layer features
        # hidden shape: (1, 32, 512) -> The updated hidden state for the NEXT time step
        
        # Predict the next word
        # We concatenate the GRU output and the context vector one more time.
        # (32, 1, 512) + (32, 1, 512) = (32, 1, 1024)
        # The linear layer (fc_out) takes this 1024-dimensional vector and outputs 
        # a score for every single word in our French dictionary (e.g., 15,000 words).
        prediction = self.fc_out(torch.cat((output, context), dim=2))
        # prediction shape: (32, 1, 15000)
        
        return prediction, hidden
    
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder        
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src shape: (32, 10) -> 32 english sentences, 10 words each
        # trg shape: (32, 12) -> The true French translations (used for Teacher Forcing)
        
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc_out.out_features
        
        # Create an empty container to hold all our predictions
        # We need a slot for every word in the target sequence, for the whole batch.
        # Shape: (32, 12, out_features)
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        
        # Run the Encoder
        # We pass the entire source sequence through the encoder all at once.
        # encoder_outputs: Memory for Attention. Shape: (32, 10, hidden_dim)
        # hidden: Context to kick-start the Decoder. Shape: (1, 32, hidden_dim)
        encoder_outputs, hidden = self.encoder(src)
        
        # Prepare the first input for the Decoder
        # trg[:, 0] grabs the very first token for every sentence in the batch.
        # In translation tasks, this is usually a special <sos> (Start Of Sentence) token.
        # We use .unsqueeze(1) to make the shape (32, 1) instead of just (32)
        input = trg[:, 0].unsqueeze(1) 
        
        # The Decoder Loop
        # We start at 1 (not 0) because we already have the <sos> token.
        # We generate the rest of the words in the target sentence.
        for t in range(1, trg_len):
    
            # Pass the current input, previous hidden state, and encoder memory to the Decoder
            output, hidden = self.decoder(input, hidden, encoder_outputs)
            
            # output shape from decoder: (32, 1, out_features)
            # We save this prediction into our giant container 'outputs' at the current time step 't'
            # .squeeze(1) changes it to (32, out_features) to fit into the container perfectly
            outputs[:, t, :] = output.squeeze(1)
            
            # Decide what the NEXT input will be
            
            # First, what was the model's actual best guess?
            # argmax(2) finds the index (integer ID) of the word with the highest score.
            # top1 shape: (32, 1) -> The model's predicted word IDs
            top1 = output.argmax(2) 
            
            # Now, do we use Teacher Forcing?
            # torch.rand(1).item() generates a random float between 0 and 1.
            # If it's less than our ratio (0.5), we cheat! We use the REAL target word (trg[:, t]).
            # If it's greater, we force the model to use its own guess (top1).
            if torch.rand(1).item() < teacher_forcing_ratio:
                input = trg[:, t].unsqueeze(1) # Teacher forcing: Give it the right answer
            else:
                input = top1 # No teacher forcing: Let it use its own guess
                
        # Shape: (32, 12, 15000)
        return outputs

#### Part 3

In [8]:
dataset = load_dataset("bentrevett/multi30k")
train_data = dataset['train']
test_data = dataset['test']

In [10]:
# Simple tokenizer (splits by space and lowercases)
def tokenize(text):
    return text.lower().split()

# Basic Vocabulary class
# It acts as a bilingual dictionary between the user and the model.
# It translates human words into integer IDs before feeding data to the model.
# It translates the model's integer ID predictions back into human words so we can read the output.

class Vocab:
    def __init__(self, frequencies, max_size=10000):
        self.itos = {0: "<pad>", 1: "<sos>", 2: "<eos>", 3: "<unk>"} # Integer-To-String
        self.stoi = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3} # String-To-Integer
        
        for i, (word, _) in enumerate(frequencies.most_common(max_size)):
            self.itos[i + 4] = word # + 4 because we already have 3 special tokens
            self.stoi[word] = i + 4
            
    def encode(self, text):
        return [1] + [self.stoi.get(w, 3) for w in tokenize(text)] + [2]
        
    def decode(self, indices):
        return [self.itos.get(idx, "<unk>") for idx in indices if idx not in (0, 1, 2)]
    
    def __len__(self):
        return len(self.itos)

In [14]:
# Count word frequencies in the dataset
en_counter = Counter()
de_counter = Counter()

In [15]:
for item in train_data:
    de_counter.update(tokenize(item['de']))
    en_counter.update(tokenize(item['en']))

In [16]:
src_vocab = Vocab(de_counter) # Source: German
trg_vocab = Vocab(en_counter) # Target: English

In [26]:
len(src_vocab)

10004

In [ ]:
len(trg_vocab)

In [20]:
# PyTorch Dataset so we can can use dataloaders
class Multi30kDataset(Dataset):
    def __init__(self, data, src_vocab, trg_vocab):
        self.data = data
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        src = self.data[idx]['de']
        trg = self.data[idx]['en']
        
        # Use the Vocab class to turn strings into tensors of integer IDs
        return torch.tensor(self.src_vocab.encode(src)), torch.tensor(self.trg_vocab.encode(trg))
    
def collate_fn(batch):
    # Separate the source and target tuples into their own lists
    src_batch, trg_batch = zip(*batch)
    
    # Pad sequences with 0 (<pad> token ID) so they form a rectangular matrix
    src_padded = nn.utils.rnn.pad_sequence(src_batch, padding_value=0, batch_first=True)
    trg_padded = nn.utils.rnn.pad_sequence(trg_batch, padding_value=0, batch_first=True)
    return src_padded, trg_padded

In [21]:
# Create DataLoaders
train_dataset = Multi30kDataset(train_data, src_vocab, trg_vocab)
test_dataset = Multi30kDataset(test_data, src_vocab, trg_vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [22]:
device = torch.device('cuda:2' if torch.cuda.is_available() else 'cpu')

In [24]:
encoder = Encoder(input_dim=len(src_vocab), emb_dim=256, hidden_dim=512)
decoder = Decoder(output_dim=len(trg_vocab), emb_dim=256, hidden_dim=512)
model = Seq2Seq(encoder, decoder, device).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
# Ignore the <pad> token ID (0) so the model doesn't calculate loss on empty padding space
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [25]:
# Training Loop
for epoch in range(10):
    model.train()
    epoch_loss = 0
    
    for src, trg in train_loader:
        src, trg = src.to(device), trg.to(device)
        
        optimizer.zero_grad()
        output = model(src, trg)
        
        # Flatten outputs and targets for the loss function
        output_dim = output.shape[-1]
        
        # We slice [:, 1:] to skip calculating loss on the <sos> token
        output = output[:, 1:].reshape(-1, output_dim) 
        trg = trg[:, 1:].reshape(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        
        # Gradient clipping to avoid exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        epoch_loss += loss.item()
        
    print(f"Epoch {epoch+1} | Train Loss: {epoch_loss / len(train_loader):.4f}")

Epoch 1 | Train Loss: 4.1639
Epoch 2 | Train Loss: 3.0503
Epoch 3 | Train Loss: 2.5326
Epoch 4 | Train Loss: 2.1735
Epoch 5 | Train Loss: 1.9568
Epoch 6 | Train Loss: 1.8239
Epoch 7 | Train Loss: 1.7179
Epoch 8 | Train Loss: 1.6349
Epoch 9 | Train Loss: 1.5710
Epoch 10 | Train Loss: 1.5050


In [29]:
def evaluate_bleu(model, dataloader, trg_vocab, device):
    model.eval()
    references = []
    hypotheses = []
    
    with torch.no_grad():
        for src, trg in dataloader:
            src = src.to(device)
            trg = trg.to(device)
            # Turn OFF teacher forcing for evaluation (force it to use its own predictions)
            output = model(src, trg, teacher_forcing_ratio=0.0)
            
            # Find the ID with the highest probability
            predictions = output.argmax(dim=-1)
            
            for i in range(trg.shape[0]):
                # Use the Vocab class to decode the IDs back into words
                ref_words = trg_vocab.decode(trg[i].tolist())
                pred_words = trg_vocab.decode(predictions[i].tolist())
                
                references.append([ref_words]) # NLTK BLEU expects reference as a list of lists
                hypotheses.append(pred_words)

    # Calculate corpus-level BLEU
    return corpus_bleu(references, hypotheses) * 100

In [30]:
# Calculate final test BLEU score
bleu_score = evaluate_bleu(model, test_loader, trg_vocab, device)
print(f"Final Test BLEU Score: {bleu_score:.2f}")

Final Test BLEU Score: 25.10


In [31]:
def translate_sentence(sentence, model, src_vocab, trg_vocab, device, max_length=50):
    model.eval()
    
    src_indexes = src_vocab.encode(sentence)
    
    # Convert to tensor and add a batch dimension of 1 -> Shape: (1, seq_len)
    src_tensor = torch.tensor(src_indexes).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Run the encoder to get the memory banks
        encoder_outputs, hidden = model.encoder(src_tensor)
        
    # Prepare the first decoder input: the <sos> token ID
    trg_indexes = [trg_vocab.stoi["<sos>"]]
    
    # Generate the translation word by word
    for _ in range(max_length):
        # Grab the very last predicted token to feed as the next input
        trg_tensor = torch.tensor([trg_indexes[-1]]).unsqueeze(0).to(device)
        
        with torch.no_grad():
            output, hidden = model.decoder(trg_tensor, hidden, encoder_outputs)
            
        # Get the ID of the word with the highest probability
        pred_token = output.argmax(2).item()
        trg_indexes.append(pred_token)
        
        # Stop generating if the model outputs the End-Of-Sentence token
        if pred_token == trg_vocab.stoi["<eos>"]:
            break
            
    # Convert the predicted IDs back to a readable English string
    translated_words = trg_vocab.decode(trg_indexes)
    
    return " ".join(translated_words)

In [35]:
german_sentence = "An dich in zweitausend oder zwanzigtausend Jahren" 
# "To you, in two thousand or twenty thousand years"

translation = translate_sentence(german_sentence, model, src_vocab, trg_vocab, device)
print(f"German: {german_sentence}")
print(f"English: {translation}")

German: An dich in zweitausend oder zwanzigtausend Jahren
English: native native american <unk> <unk> robes
